# 03 — Validate 3D vs Bootstrap

Compare the bootstrapped GW spectrum from notebook 02 against an actual
sledgehamr 3D run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ptbuilder as pt
from pathlib import Path

## 1. Load bootstrapped spectrum (from notebook 02)

In [ ]:
lambda_bar = 0.84
potential  = pt.Phi4Potential(lambda_bar)
config     = pt.Config()
model      = pt.PhysicsModel(potential, config)

scan_results = pt.load_scan(model.results_dir / 'scan_cache.h5')
times        = next(iter(scan_results.values())).times

# Rebuild bubble population (use same seed as notebook 02)
bubble_pop = pt.BubblePopulation.generate(
    n_bubbles  = 64,
    nucleation = pt.ExponentialNucleation(1., 0., 10.),
    kinematics = model.kinematics,
    box_size   = 200.,
    seed       = 42,
)

weights, pair_i, pair_j = pt.load_weights(
    model.results_dir / 'weights_out_seed42.h5',
    n_t = len(times),
)

k_out = np.logspace(-2, 1, 200)
k_boot, spec_boot = pt.bootstrap_spectrum(
    scan_results, bubble_pop, weights, pair_i, pair_j,
    t_range    = (times[0], times[-1]),
    k_out      = k_out,
    lambda_bar = lambda_bar,
)
print('Bootstrap spectrum loaded.')

## 2. Load sledgehamr 3D output

Point `sledgehamr_path` to the output directory of your 3D run.
The box size `L` must match what was used in the simulation.

In [ ]:
sledgehamr_path = '/path/to/sledgehamr/output'  # <-- set this
L               = 200.    # box side length
zero_pad        = 1.

try:
    slout = pt.load_output(sledgehamr_path)
    k_3d, spec_3d, _ = pt.get_final_spectrum(slout, L=L, zero_pad=zero_pad, n_avg=5)
    has_3d = True
    print('3D spectrum loaded.')
except Exception as e:
    has_3d = False
    print(f'3D output not loaded: {e}')

## 3. Compare

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(k_boot, spec_boot, color='black', label='Bootstrap (2D scan)')

if has_3d:
    ax.plot(k_3d, spec_3d, color='red', label='sledgehamr (3D)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('k')
ax.set_ylabel(r'$\mathrm{d}E_{\mathrm{GW}}/\mathrm{d}\ln k$')
ax.set_title('Bootstrap vs 3D GW spectrum')
ax.legend()
plt.tight_layout()

## 4. Quantitative comparison

In [ ]:
if has_3d:
    # Resample bootstrap to 3D k grid for comparison
    spec_boot_resampled = np.interp(k_3d, k_boot, spec_boot, left=0., right=0.)

    mask   = (k_3d > 0) & (spec_3d > 0)
    ratio  = spec_3d[mask] / spec_boot_resampled[mask]
    print(f'Mean ratio (3D / bootstrap): {np.mean(ratio):.3f}')
    print(f'Std  ratio:                  {np.std(ratio):.3f}')
else:
    print('No 3D data available for comparison.')